# 🏆 Mini Project: Scouting the Academy's Next Star (NumPy)

### The Scenario

You are a **data analyst for a sports academy**. The head coach is putting together a **6-player squad** for an upcoming regional tournament, and has given you a fixed set of rules to follow so every analyst on the team produces a consistent, comparable report.

You have performance data for 12 players across 4 physical metrics. Follow the steps below exactly — the weights and selection rule are fixed by the coaching staff, so your final squad should match a single correct answer. Your own input comes in at the very end, in how you *explain* the result.

**Deliverable:** a ranked, structured record of players, the final 6-player squad (following the fixed rule below), and a short written justification.


## 1. The Data

Run the cell below — this is your starting dataset. Don't change it directly; if you need to adjust anything, work on a copy.

| Column | Meaning |
|---|---|
| 0 | Sprint Speed (km/h) |
| 1 | Endurance Score (0–100) |
| 2 | Strength Score (0–100) |
| 3 | Agility Score (0–100) |


In [7]:
import numpy as np

players = np.array([
    "Adam", "Mostafa", "Yara", "Hana", "Ziad", "Malak",
    "Omar", "Farida", "Karim", "Nada", "Tarek", "Reem"
])

positions = np.array([
    "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper", "Defender",
    "Midfielder", "Forward", "Defender", "Midfielder", "Forward", "Goalkeeper"
])

# 12 players x 4 metrics: [Sprint Speed, Endurance, Strength, Agility]
performance = np.array([
    [28.5, 82, 75, 88],
    [26.0, 90, 88, 70],
    [30.2, 78, 60, 92],
    [27.8, 85, 70, 80],
    [24.5, 95, 92, 65],
    [25.9, 70, 65, 75],
    [29.1, 80, 55, 90],
    [31.0, 60, 50, 95],
    [23.8, 88, 90, 60],
    [28.9, 83, 68, 85],
    [27.2, 76, 72, 78],
    [22.9, 93, 95, 58],
])

print(players.shape, positions.shape, performance.shape)

(12,) (12,) (12, 4)


## 2. Explore & Clean

Before ranking anyone, get familiar with the data — and fix a known issue.

1. Look at the shape and structure of `performance`. Pick a couple of players and compare their rows by eye using indexing.
2. There is a known data-entry glitch: **any Strength score above 90 should be capped at exactly 90.** Fix this on a **copy** of `performance` (call it `performance_clean`), so you don't lose the original data.

   💡 `np.clip(array, min, max)` caps values into a range — pass `None` for a bound you don't want to limit.


In [8]:
# Explore the data, then create performance_clean here
print("Original performance shape:", performance.shape)
print("Player 0 (Adam) original stats:", performance[0])
print("Player 11 (Reem) original stats:", performance[11])

performance_clean = performance.copy()

performance_clean[:, 2] = np.clip(performance_clean[:, 2], None, 90)

print("\nPlayer 4 (Ziad) clean stats (Strength capped at 90):", performance_clean[4])
print("Player 11 (Reem) clean stats (Strength capped at 90):", performance_clean[11])

Original performance shape: (12, 4)
Player 0 (Adam) original stats: [28.5 82.  75.  88. ]
Player 11 (Reem) original stats: [22.9 93.  95.  58. ]

Player 4 (Ziad) clean stats (Strength capped at 90): [24.5 95.  90.  65. ]
Player 11 (Reem) clean stats (Strength capped at 90): [22.9 93.  90.  58. ]


## 3. Build the Talent Score

The coaching staff has already agreed on how much each metric should count toward a player's overall Talent Score, so every scout's numbers line up. This tournament is a fast-paced format, so Sprint Speed and Agility are weighted highest:

| Metric | Weight |
|---|---|
| Sprint Speed | ×1.5 |
| Endurance | ×0.7 |
| Strength | ×0.5 |
| Agility | ×1.3 |

Combine `performance_clean` with this weight array using **broadcasting**, then sum across the metrics (per player) to get one `talent_score` value per player.


In [9]:
weights = np.array([1.5, 0.7, 0.5, 1.3])  


talent_score = np.sum(performance_clean * weights, axis=1)

print("Talent Scores for all players:")
print(np.round(talent_score, 2))

Talent Scores for all players:
[252.05 237.   249.5  240.2  232.75 217.85 244.15 237.   220.3  245.95
 231.4  219.85]


## 4. Package Everything into a Structured Array

Right now you have three separate arrays (`players`, `positions`, `talent_score`) that all describe the same 12 people — that's exactly the situation structured arrays are built for.

1. Build a structured array called `Squad` combining all three, with fields: `name` (string), `position` (string), and `score` (float).

   💡 One way: build it from a list of tuples using `dtype=[("name","U10"), ("position","U12"), ("score","f4")]`. You can construct the list of tuples with a loop over the index, e.g. `[(players[i], positions[i], talent_score[i]) for i in range(len(players))]`.

2. Explore it: print the field names, the shape, and confirm you can access `Squad["name"]` and `Squad["score"]` correctly.


In [10]:
# Build the Squad structured array her
dt = np.dtype([("name", "U10"), ("position", "U12"), ("score", "f4")])

squad_data = [(players[i], positions[i], talent_score[i]) for i in range(len(players))]

Squad = np.array(squad_data, dtype=dt)

print("Field Names:", Squad.dtype.names)
print("Shape:", Squad.shape)
print("\nFirst 3 players' names:", Squad["name"][:3])
print("First 3 players' scores:", Squad["score"][:3])

Field Names: ('name', 'position', 'score')
Shape: (12,)

First 3 players' names: ['Adam' 'Mostafa' 'Yara']
First 3 players' scores: [252.05 237.   249.5 ]


## 5. Filter & Sort by Field

Use field-based access to answer these:

1. Which players are Goalkeepers? (filter `Squad` where `position == "Goalkeeper"`)
2. Sort the whole `Squad` array by `score`, from highest to lowest, and print the result. (`np.sort` sorts ascending by default — think about how to reverse it.)
3. Convert `Squad` to a **record array** and use dot notation (`.name`, `.score`) to print the top 3 names and scores from your sorted result.


In [11]:
# Filter, sort, and use a record array here

goalkeepers = Squad[Squad['position'] == 'Goalkeeper']
print("Goalkeepers:\n", goalkeepers)

Squad_sorted = np.sort(Squad, order='score')[::-1]
print("\nSorted Squad (Top 3 shown):\n", Squad_sorted[:3])

Squad_rec = Squad_sorted.view(np.recarray)
print("\nTop 3 Players (using Record Array):")
for i in range(3):
    print(f"{i+1}. {Squad_rec.name[i]} - Score: {Squad_rec.score[i]:.2f}")

Goalkeepers:
 [('Ziad', 'Goalkeeper', 232.75) ('Reem', 'Goalkeeper', 219.85)]

Sorted Squad (Top 3 shown):
 [('Adam', 'Forward', 252.05) ('Yara', 'Midfielder', 249.5 )
 ('Nada', 'Midfielder', 245.95)]

Top 3 Players (using Record Array):
1. Adam - Score: 252.05
2. Yara - Score: 249.50
3. Nada - Score: 245.95


## 6. Select the Squad

Apply this fixed selection rule, exactly as written, so every scout arrives at the same squad:

1. Take the **6 players with the highest `score`**.
2. **Check the rule:** the squad must include **at least 1 Goalkeeper**.
3. **If it doesn't:** remove the *lowest-scoring* player currently in the top 6, and replace them with the *highest-scoring* Goalkeeper (even if that Goalkeeper's score is outside the top 6). **If it already does, leave the squad as is** — don't swap anyone.
4. Print your final 6-player squad (name, position, score).
5. Sanity check: print the average `score` of your final squad vs. the average `score` of the players left out.


In [12]:
# Select your final squad and sanity-check it here

final_squad = Squad_sorted[:6].copy()

if 'Goalkeeper' not in final_squad['position']:
    print("\n[Rule Triggered]: No Goalkeeper in Top 6. Swapping lowest player for best GK...")
    
    best_gk = Squad_sorted[Squad_sorted['position'] == 'Goalkeeper'][0]
    
    final_squad[-1] = best_gk

print("\n--- Final 6-Player Squad ---")
for player in final_squad:
    print(f"{player['name']:<10} | {player['position']:<12} | {player['score']:.2f}")

left_out_players = np.setdiff1d(Squad_sorted, final_squad)
avg_squad_score = np.mean(final_squad['score'])
avg_left_out_score = np.mean(left_out_players['score'])

print(f"\nAverage Score of Final Squad: {avg_squad_score:.2f}")
print(f"Average Score of Left-Out Players: {avg_left_out_score:.2f}")


[Rule Triggered]: No Goalkeeper in Top 6. Swapping lowest player for best GK...

--- Final 6-Player Squad ---
Adam       | Forward      | 252.05
Yara       | Midfielder   | 249.50
Nada       | Midfielder   | 245.95
Omar       | Midfielder   | 244.15
Hana       | Forward      | 240.20
Ziad       | Goalkeeper   | 232.75

Average Score of Final Squad: 244.10
Average Score of Left-Out Players: 227.23


## 7. Final Report

In 4–6 sentences, write a short summary as if you were sending it to the coach. Include:
- The final squad (names)
- One sentence on why Endurance was weighted highest (in your own words, based on what a tournament format demands)
- Which player was swapped in/out to satisfy the Goalkeeper rule (if applicable), and what that costs the squad in terms of average score

*(Write this as a markdown cell or as comments — either is fine.)*


# Final Coach Report

> **Objective:** Finalize the 6-player squad based on weighted performance metrics and positional requirements.

### Final Squad Selection
* **Adam**
* **Yara**
* **Nada**
* **Omar**
* **Hana**
* **Ziad** *(Goalkeeper)*

---

### Metric Weighting Rationale
In a fast-paced regional tournament format, maintaining high output is critical. While physical attributes like **Sprint Speed** and **Agility** heavily dictate quick plays, combining them with solid **Endurance** ensures players do not burn out mid-match.

---

### Goalkeeper Rule Adjustment
The initial top 6 players did not include a Goalkeeper based purely on raw talent scores. To satisfy the squad requirements:

* **Swapped Out:** Farida *(Forward, Score: 237.0)*
* **Swapped In:** Ziad *(Goalkeeper, Score: 232.75)*

**Impact:** This necessary swap only cost the team a marginal **0.71 points** in the average overall squad score. This preserves the team's highly competitive baseline while strictly securing our net.